# Data Cleaning — AI Agent for Cybersecurity Incident ResponseThis notebook cleans the raw incident data collected in Task 1 and prepares it for storage (Task 3).**What's being cleaned:**- Duplicate incident records- Inconsistent text casing (e.g. "high" vs "High")- Missing values in status and resolution time- Extra whitespace in fields- Invalid time values**Tools used:** Python, Pandas

## 1. Import libraries

In [1]:
import pandas as pdimport numpy as nppd.set_option('display.max_columns', None)pd.set_option('display.width', 150)

## 2. Load the raw data

In [2]:
df = pd.read_csv('../data_raw/incidents_db_raw.csv')print("Shape before cleaning:", df.shape)df.head(10)

Shape before cleaning: (10, 13)


## 3. Initial inspectionLet's check for the usual issues — nulls, duplicates, and weird values.

In [3]:
print("Missing values per column:")print(df.isnull().sum())print()print("Duplicate rows:", df.duplicated().sum())

Missing values per column:
incident_id            0
date                   0
time                   0
incident_type          0
severity               0
affected_system        0
source_ip              0
destination_ip         0
status                 1
response_action        0
analyst_name           0
resolution_time_hrs    1
notes                  0
dtype: int64

Duplicate rows: 1


In [4]:
# check unique values in columns that should be consistent categoriesprint("Unique severity values:", df['severity'].unique())print("Unique incident_type values:", df['incident_type'].unique())print("Unique time values (look for anything invalid):")print(df['time'].unique())

Unique severity values: <StringArray>
['High', 'Critical', 'high', 'Medium']
Length: 4, dtype: str
Unique incident_type values: <StringArray>
['Phishing', 'malware', 'DDoS', 'Unauthorized Access', 'Data Exfiltration', 'SQL Injection ', 'Insider Threat', 'Zero-Day']
Length: 8, dtype: str
Unique time values (look for anything invalid):
<StringArray>
['09:23', '14:05', '11:45', '08:12', '16:30', '10:12', '07:50', '13:20', '99:99']
Length: 9, dtype: str


## 4. Remove duplicate rowsWe found a duplicate of INC-003. Keeping only the first occurrence.

In [5]:
before = len(df)df = df.drop_duplicates()after = len(df)print(f"Removed {before - after} duplicate row(s)")

Removed 1 duplicate row(s)


## 5. Fix inconsistent text casing`severity` had both "high" and "High" — these should be treated as the same value.Also trimming whitespace from `incident_type` (e.g. "SQL Injection " had a trailing space).**Note:** `.str.title()` on its own breaks acronyms like DDoS and SQL (turns them into "Ddos" and "Sql Injection"). Fixed this with a small manual correction map after applying title case.

In [6]:
df['severity'] = df['severity'].str.strip().str.title()# .title() breaks acronyms like DDoS and SQL (turns them into Ddos, Sql)# so fix incident_type with a manual correction map instead of relying on title() alonedf['incident_type'] = df['incident_type'].str.strip().str.title()acronym_fixes = {    'Ddos': 'DDoS',    'Sql Injection': 'SQL Injection'}df['incident_type'] = df['incident_type'].replace(acronym_fixes)df['status'] = df['status'].str.strip()df['analyst_name'] = df['analyst_name'].str.strip()print("Severity values after fix:", df['severity'].unique())print("Incident type values after fix:", df['incident_type'].unique())

Severity values after fix: <StringArray>
['High', 'Critical', 'Medium']
Length: 3, dtype: str
Incident type values after fix: <StringArray>
['Phishing', 'Malware', 'DDoS', 'Unauthorized Access', 'Data Exfiltration', 'SQL Injection', 'Insider Threat', 'Zero-Day']
Length: 8, dtype: str


## 6. Handle missing valuesINC-005 has a blank `status` and blank `resolution_time_hrs`. Since the incident notes say forensics just started, it makes sense to fill these in logically rather than dropping the row.

In [7]:
print(df[df['status'].isnull() | df['resolution_time_hrs'].isnull()])

  incident_id        date   time      incident_type  severity affected_system     source_ip destination_ip status  \
5     INC-005  2026-02-10  16:30  Data Exfiltration  Critical     File Server  192.168.10.5   45.33.32.156    NaN   

                     response_action analyst_name  resolution_time_hrs                                    notes  
5  network blocked forensics started      Neha M.                  NaN  large file transfer flagged by firewall  


In [8]:
# Fill missing status with 'Under Review' since the incident is still activedf['status'] = df['status'].fillna('Under Review')# Fill missing resolution time with 0 (not resolved yet)df['resolution_time_hrs'] = df['resolution_time_hrs'].fillna(0).astype(int)print("Missing values remaining:")print(df.isnull().sum())

Missing values remaining:
incident_id            0
date                   0
time                   0
incident_type          0
severity               0
affected_system        0
source_ip              0
destination_ip         0
status                 0
response_action        0
analyst_name           0
resolution_time_hrs    0
notes                  0
dtype: int64


## 7. Fix invalid time valueINC-009 has time `99:99` which is not a valid time. Since we don't know the real time, we'll flag it for manual review rather than guessing.

In [9]:
def is_valid_time(t):    try:        h, m = t.split(':')        return 0 <= int(h) <= 23 and 0 <= int(m) <= 59    except:        return Falsedf['time_valid'] = df['time'].apply(is_valid_time)print(df[~df['time_valid']][['incident_id', 'time']])

  incident_id   time
9     INC-009  99:99


In [10]:
# Replace invalid time with 'Unknown' instead of guessing a valuedf.loc[~df['time_valid'], 'time'] = 'Unknown'df = df.drop(columns=['time_valid'])print("Time values now:", df['time'].unique())

Time values now: <StringArray>
['09:23', '14:05', '11:45', '08:12', '16:30', '10:12', '07:50', '13:20', 'Unknown']
Length: 9, dtype: str


## 8. Validate data consistency across fieldsQuick sanity checks — making sure severity levels and incident types only contain expected values.

In [11]:
expected_severities = {'Critical', 'High', 'Medium', 'Low'}unexpected = set(df['severity'].unique()) - expected_severitiesprint("Unexpected severity values found:", unexpected if unexpected else "None — all good")print()print("Final incident type counts:")print(df['incident_type'].value_counts())

Unexpected severity values found: None — all good

Final incident type counts:
incident_type
Phishing               2
Malware                1
DDoS                   1
Unauthorized Access    1
Data Exfiltration      1
SQL Injection          1
Insider Threat         1
Zero-Day               1
Name: count, dtype: int64


## 9. Final check before saving

In [12]:
print("Shape after cleaning:", df.shape)df.head(10)

Shape after cleaning: (9, 13)


## 10. Save the cleaned dataset

In [13]:
df.to_csv('../data/incidents_db_cleaned.csv', index=False)print("Saved cleaned file to data/incidents_db_cleaned.csv")

Saved cleaned file to data/incidents_db_cleaned.csv


## Summary — Before vs After| Check | Before | After ||---|---|---|| Total rows | 10 | 9 || Duplicate rows | 1 | 0 || Missing status | 1 | 0 || Missing resolution_time_hrs | 1 | 0 || Inconsistent severity casing | Yes ("high" vs "High") | Fixed — all Title Case || Invalid time values | 1 (`99:99`) | Replaced with "Unknown" || Trailing whitespace in text fields | Yes | Removed |This cleaned file (`incidents_db_cleaned.csv`) is what gets used in Task 3 for loading into a database.